# Flight Delay Data Quality Profile

Notebook dùng PySpark để profile **typed raw Parquet**: schema, inventory, completeness, validity, uniqueness, cardinality, referential integrity, target, leakage, scorecard và handoff. Dữ liệu vẫn là typed raw; không cleaning, feature engineering hay model training.

## 1. Runtime & Configuration

Kiểm tra runtime và ba thư mục đầu vào trước khi profiling. Đặt `FLIGHT_TYPED_PARQUET_DIR` trong môi trường chạy; driver và worker dùng cùng Python của kernel.

**Nếu kernel VS Code báo biến môi trường chưa được đặt:** biến được export ở process khác không tự truyền vào kernel hiện tại. Chạy đoạn dưới trong một cell tạm **trước cell cấu hình**, với working directory là repo root hoặc thư mục `notebooks`. Đoạn này chỉ đặt biến cho kernel hiện tại, không ghi dữ liệu và không ghi đè cấu hình đã có:

```python
import os
from pathlib import Path

if not os.environ.get("FLIGHT_TYPED_PARQUET_DIR", "").strip():
    cwd = Path.cwd().resolve()
    repo_root = next(
        (p for p in (cwd, *cwd.parents)
         if (p / "notebooks").is_dir() and (p / "data" / "typed-parquet").is_dir()),
        None,
    )
    if repo_root is None:
        raise RuntimeError("Hãy mở notebook trong repository hoặc tự đặt FLIGHT_TYPED_PARQUET_DIR.")
    os.environ["FLIGHT_TYPED_PARQUET_DIR"] = str(repo_root / "data" / "typed-parquet")
```

Sau đó chạy lại từ cell cấu hình. Khi restart kernel, cần thiết lập lại biến nếu process khởi chạy VS Code chưa có biến này. Notebook vẫn lấy đường dẫn đầu vào từ `os.environ`.


In [1]:
import os, sys, platform, json
from pathlib import Path
import pyspark
from pyspark.sql import SparkSession, functions as F, types as T
from IPython.display import display, Markdown
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
if not os.environ.get('FLIGHT_TYPED_PARQUET_DIR', '').strip():
    raise RuntimeError(
        'FLIGHT_TYPED_PARQUET_DIR is not set in this kernel. '
        'Run the setup snippet in section 1 before this cell.'
    )
root = Path(os.environ['FLIGHT_TYPED_PARQUET_DIR']).expanduser().resolve()
missing_dirs = [name for name in ['flights', 'airports', 'airlines'] if not (root / name).is_dir()]
if missing_dirs:
    raise FileNotFoundError(f'Typed Parquet directories missing under {root}: {missing_dirs}')
print('Python:', platform.python_version())
print('Python executable:', sys.executable)
print('PySpark:', pyspark.__version__)
spark = (SparkSession.builder.master('local[2]').appName('FlightDataQualityProfile')
         .config('spark.sql.shuffle.partitions', '12')
         .config('spark.ui.enabled', 'false').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Spark:', spark.version)
warnings = []
def table(rows):
    if not rows:
        print('(no rows)'); return
    keys = list(rows[0])
    def esc(v): return str(v).replace('|', '/').replace('\n', ' ')
    display(Markdown('| ' + ' | '.join(keys) + ' |\n| ' + ' | '.join(['---']*len(keys)) + ' |\n' +
                     '\n'.join('| ' + ' | '.join(esc(r[k]) for k in keys) + ' |' for r in rows)))
def reference_check(name, observed, expected, tolerance=0):
    ok = abs(observed - expected) <= tolerance
    print(f'{name}: observed={observed}; reference={expected}; status={"MATCH" if ok else "DISCREPANCY"}')
    if not ok: warnings.append(f'{name}: {observed} vs reference {expected}')
def n_if(condition):
    return F.count(F.when(condition, 1))

Python: 3.11.16
Python executable: /home/admins/miniforge3/envs/bigdata-spark/bin/python
PySpark: 3.5.9


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/08 18:26:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.9


## 2. Load Typed Parquet

Đọc trực tiếp ba bảng typed Parquet. Không cache toàn bảng để giữ mức dùng bộ nhớ thấp; Spark chỉ quét các cột cần cho từng aggregate.

In [2]:
flights = spark.read.parquet(str(root / 'flights'))
airports = spark.read.parquet(str(root / 'airports'))
airlines = spark.read.parquet(str(root / 'airlines'))
tables = {'flights': flights, 'airports': airports, 'airlines': airlines}

## 3. Basic Inventory

Đếm thực tế và in schema để đối chiếu quy mô, kiểu dữ liệu. Chỉ hiển thị tối đa 5 dòng mẫu cho mỗi bảng; reference không thay thế kết quả đo.

In [3]:
counts = {name: df.count() for name, df in tables.items()}
table([{'table': name, 'rows': counts[name], 'columns': len(df.columns)} for name, df in tables.items()])
for name, expected in [('flights', 5819079), ('airports', 322), ('airlines', 14)]:
    reference_check(name + '_rows', counts[name], expected)
    print(name)
    tables[name].printSchema()
    tables[name].show(5, truncate=24)
N = counts['flights']

| table | rows | columns |
| --- | --- | --- |
| flights | 5819079 | 31 |
| airports | 322 | 7 |
| airlines | 14 | 2 |

flights_rows: observed=5819079; reference=5819079; status=MATCH
flights
root
 |-- YEAR: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: string (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: string (nullable = true)
 |-- DEPARTURE_TIME: string (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: string (nullable = true)
 |-- ARRIVAL_TIME: string (nullable = true)
 |-- A

+----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+-----+
|YEAR|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|MONTH|
+----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-------

+---------+------------------------+-----------+-----+-------+--------+----------+
|IATA_CODE|                 AIRPORT|       CITY|STATE|COUNTRY|LATITUDE| LONGITUDE|
+---------+------------------------+-----------+-----+-------+--------+----------+
|      ABE|Lehigh Valley Interna...|  Allentown|   PA|    USA|40.65236|  -75.4404|
|      ABI|Abilene Regional Airport|    Abilene|   TX|    USA|32.41132|  -99.6819|
|      ABQ|Albuquerque Internati...|Albuquerque|   NM|    USA|35.04022|-106.60919|
|      ABR|Aberdeen Regional Air...|   Aberdeen|   SD|    USA|45.44906| -98.42183|
|      ABY|Southwest Georgia Reg...|     Albany|   GA|    USA|31.53552| -84.19447|
+---------+------------------------+-----------+-----+-------+--------+----------+
only showing top 5 rows

airlines_rows: observed=14; reference=14; status=MATCH
airlines
root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)

+---------+----------------------+
|IATA_CODE|               AIRLINE|
+-------

## 4. Completeness

Tính null cho mọi cột; blank là chuỗi không null chỉ gồm khoảng trắng hoặc rỗng. Hai loại được đếm riêng, tỷ lệ lấy trên tổng số dòng của bảng. Không thay thế giá trị thiếu.

In [4]:
def profile_nulls(df, total):
    expressions = []
    for field in df.schema:
        c = F.col(field.name)
        expressions.append(n_if(c.isNull()).alias(field.name + '__null'))
        if isinstance(field.dataType, T.StringType):
            expressions.append(n_if(c.isNotNull() & (F.trim(c) == '')).alias(field.name + '__blank'))
    agg = df.agg(*expressions).first().asDict()
    rows = []
    for field in df.schema:
        nc = agg[field.name + '__null']; bc = agg.get(field.name + '__blank')
        rows.append({'column': field.name, 'null_count': nc, 'null_rate_pct': 100*nc/total,
                     'blank_count': bc if bc is not None else 'N/A',
                     'blank_rate_pct': 100*bc/total if bc is not None else 'N/A'})
    return rows
completeness = {name: profile_nulls(df, counts[name]) for name, df in tables.items()}
for name, rows in completeness.items():
    print(name); table(rows)
flight_missing = {r['column']: r for r in completeness['flights']}
reference_check('arrival_delay_null', flight_missing['ARRIVAL_DELAY']['null_count'], 105071)
coord = airports.agg(
    n_if(F.col('LATITUDE').isNull() & F.col('LONGITUDE').isNull()).alias('both_null'),
    n_if(F.col('LATITUDE').isNull() != F.col('LONGITUDE').isNull()).alias('one_null')).first().asDict()
reference_check('airport_both_coordinates_null', coord['both_null'], 3)

flights


| column | null_count | null_rate_pct | blank_count | blank_rate_pct |
| --- | --- | --- | --- | --- |
| YEAR | 0 | 0.0 | N/A | N/A |
| DAY | 0 | 0.0 | N/A | N/A |
| DAY_OF_WEEK | 0 | 0.0 | N/A | N/A |
| AIRLINE | 0 | 0.0 | 0 | 0.0 |
| FLIGHT_NUMBER | 0 | 0.0 | 0 | 0.0 |
| TAIL_NUMBER | 14721 | 0.25297817747447665 | 0 | 0.0 |
| ORIGIN_AIRPORT | 0 | 0.0 | 0 | 0.0 |
| DESTINATION_AIRPORT | 0 | 0.0 | 0 | 0.0 |
| SCHEDULED_DEPARTURE | 0 | 0.0 | 0 | 0.0 |
| DEPARTURE_TIME | 86153 | 1.4805263857046793 | 0 | 0.0 |
| DEPARTURE_DELAY | 86153 | 1.4805263857046793 | N/A | N/A |
| TAXI_OUT | 89047 | 1.5302593417274453 | N/A | N/A |
| WHEELS_OFF | 89047 | 1.5302593417274453 | 0 | 0.0 |
| SCHEDULED_TIME | 6 | 0.00010310910025452482 | N/A | N/A |
| ELAPSED_TIME | 105071 | 1.8056293788071962 | N/A | N/A |
| AIR_TIME | 105071 | 1.8056293788071962 | N/A | N/A |
| DISTANCE | 0 | 0.0 | N/A | N/A |
| WHEELS_ON | 92513 | 1.5898220319744756 | 0 | 0.0 |
| TAXI_IN | 92513 | 1.5898220319744756 | N/A | N/A |
| SCHEDULED_ARRIVAL | 0 | 0.0 | 0 | 0.0 |
| ARRIVAL_TIME | 92513 | 1.5898220319744756 | 0 | 0.0 |
| ARRIVAL_DELAY | 105071 | 1.8056293788071962 | N/A | N/A |
| DIVERTED | 0 | 0.0 | N/A | N/A |
| CANCELLED | 0 | 0.0 | N/A | N/A |
| CANCELLATION_REASON | 5729195 | 98.45535693878705 | 0 | 0.0 |
| AIR_SYSTEM_DELAY | 4755640 | 81.72496025573807 | N/A | N/A |
| SECURITY_DELAY | 4755640 | 81.72496025573807 | N/A | N/A |
| AIRLINE_DELAY | 4755640 | 81.72496025573807 | N/A | N/A |
| LATE_AIRCRAFT_DELAY | 4755640 | 81.72496025573807 | N/A | N/A |
| WEATHER_DELAY | 4755640 | 81.72496025573807 | N/A | N/A |
| MONTH | 0 | 0.0 | N/A | N/A |

airports


| column | null_count | null_rate_pct | blank_count | blank_rate_pct |
| --- | --- | --- | --- | --- |
| IATA_CODE | 0 | 0.0 | 0 | 0.0 |
| AIRPORT | 0 | 0.0 | 0 | 0.0 |
| CITY | 0 | 0.0 | 0 | 0.0 |
| STATE | 0 | 0.0 | 0 | 0.0 |
| COUNTRY | 0 | 0.0 | 0 | 0.0 |
| LATITUDE | 3 | 0.9316770186335404 | N/A | N/A |
| LONGITUDE | 3 | 0.9316770186335404 | N/A | N/A |

airlines


| column | null_count | null_rate_pct | blank_count | blank_rate_pct |
| --- | --- | --- | --- | --- |
| IATA_CODE | 0 | 0.0 | 0 | 0.0 |
| AIRLINE | 0 | 0.0 | 0 | 0.0 |

arrival_delay_null: observed=105071; reference=105071; status=MATCH


airport_both_coordinates_null: observed=3; reference=3; status=MATCH


## 5. Missing Pattern Summary

So sánh target thiếu với hai cờ hủy/chuyển hướng bằng bảng chéo và kiểm tra hai chiều. Với delay-cause, đo non-null nằm ở nhóm đến trễ ≥15 phút; đây là bằng chứng liên hệ với outcome, không chứng minh nguyên nhân missing.

In [5]:
arrival_null = F.col('ARRIVAL_DELAY').isNull()
flagged = (F.col('CANCELLED') == 1) | (F.col('DIVERTED') == 1)
flights.groupBy(arrival_null.alias('arrival_delay_null'), 'CANCELLED', 'DIVERTED').count().orderBy('arrival_delay_null', 'CANCELLED', 'DIVERTED').show(20)
patterns = flights.agg(
    n_if(arrival_null).alias('arrival_delay_null'),
    n_if(F.col('CANCELLED') == 1).alias('cancelled'),
    n_if(F.col('DIVERTED') == 1).alias('diverted'),
    n_if(arrival_null & ~flagged).alias('null_without_flag'),
    n_if(~arrival_null & flagged).alias('flag_without_null'),
    n_if(F.col('CANCELLED').isNull() | F.col('DIVERTED').isNull()).alias('unknown_flags')
).first().asDict()
table([patterns])
for key, ref in [('arrival_delay_null',105071), ('cancelled',89884), ('diverted',15187)]:
    reference_check(key, patterns[key], ref)
causes = ['AIR_SYSTEM_DELAY','SECURITY_DELAY','AIRLINE_DELAY','LATE_AIRCRAFT_DELAY','WEATHER_DELAY']
exprs = []
for c in causes:
    exprs += [n_if(F.col(c).isNotNull()).alias(c+'_present'),
              n_if(F.col(c).isNotNull() & (F.col('ARRIVAL_DELAY') >= 15)).alias(c+'_ge15')]
cause_agg = flights.agg(*exprs).first().asDict()
cause_rows = []
for c in causes:
    present = cause_agg[c+'_present']; ge15 = cause_agg[c+'_ge15']
    cause_rows.append({'column':c, 'missing_rate_pct':flight_missing[c]['null_rate_pct'],
                       'non_null':present, 'non_null_and_arrival_ge15':ge15,
                       'non_null_other_or_unknown_arrival':present-ge15,
                       'ge15_share_of_non_null_pct':100*ge15/present if present else None})
    reference_check(c+'_missing_pct', flight_missing[c]['null_rate_pct'], 81.7, 0.1)
table(cause_rows)

+------------------+---------+--------+-------+
|arrival_delay_null|CANCELLED|DIVERTED|  count|
+------------------+---------+--------+-------+
|             false|        0|       0|5714008|
|              true|        0|       1|  15187|
|              true|        1|       0|  89884|
+------------------+---------+--------+-------+



| arrival_delay_null | cancelled | diverted | null_without_flag | flag_without_null | unknown_flags |
| --- | --- | --- | --- | --- | --- |
| 105071 | 89884 | 15187 | 0 | 0 | 0 |

arrival_delay_null: observed=105071; reference=105071; status=MATCH
cancelled: observed=89884; reference=89884; status=MATCH
diverted: observed=15187; reference=15187; status=MATCH


AIR_SYSTEM_DELAY_missing_pct: observed=81.72496025573807; reference=81.7; status=MATCH
SECURITY_DELAY_missing_pct: observed=81.72496025573807; reference=81.7; status=MATCH
AIRLINE_DELAY_missing_pct: observed=81.72496025573807; reference=81.7; status=MATCH
LATE_AIRCRAFT_DELAY_missing_pct: observed=81.72496025573807; reference=81.7; status=MATCH
WEATHER_DELAY_missing_pct: observed=81.72496025573807; reference=81.7; status=MATCH


| column | missing_rate_pct | non_null | non_null_and_arrival_ge15 | non_null_other_or_unknown_arrival | ge15_share_of_non_null_pct |
| --- | --- | --- | --- | --- | --- |
| AIR_SYSTEM_DELAY | 81.72496025573807 | 1063439 | 1063439 | 0 | 100.0 |
| SECURITY_DELAY | 81.72496025573807 | 1063439 | 1063439 | 0 | 100.0 |
| AIRLINE_DELAY | 81.72496025573807 | 1063439 | 1063439 | 0 | 100.0 |
| LATE_AIRCRAFT_DELAY | 81.72496025573807 | 1063439 | 1063439 | 0 | 100.0 |
| WEATHER_DELAY | 81.72496025573807 | 1063439 | 1063439 | 0 | 100.0 |

## 6. Uniqueness & Duplicate

Kiểm tra chính xác khóa lookup; distinct bỏ qua null, missing gồm null/blank. Duplicate key count là số khóa không thiếu xuất hiện nhiều lần. Full-row flights: **STATUS = SKIPPED**, **REASON = exact full-row duplicate was previously profiled on raw data** (theo bối cảnh bàn giao); notebook không xác nhận PASS và tránh shuffle toàn bộ 31 cột.

In [6]:
uniqueness = []
for name in ['airports','airlines']:
    df = tables[name]; c = F.col('IATA_CODE')
    stats = df.agg(F.countDistinct(c).alias('distinct'), n_if(c.isNull() | (F.trim(c)=='')).alias('missing')).first().asDict()
    duplicates = df.where(c.isNotNull() & (F.trim(c) != '')).groupBy('IATA_CODE').count().where(F.col('count') > 1).count()
    uniqueness.append({'table':name, 'rows':counts[name], **stats, 'duplicate_key_count':duplicates})
table(uniqueness)
full_duplicate_status = 'SKIPPED'
print('STATUS = SKIPPED; REASON = exact full-row duplicate was previously profiled on raw data')

| table | rows | distinct | missing | duplicate_key_count |
| --- | --- | --- | --- | --- |
| airports | 322 | 322 | 0 | 0 |
| airlines | 14 | 14 | 0 | 0 |

STATUS = SKIPPED; REASON = exact full-row duplicate was previously profiled on raw data


## 7. Validity & Domain

Đếm null riêng với vi phạm domain non-null. Kiểm tra tọa độ nằm trong giới hạn địa lý; không suy ra độ chính xác vị trí. Domain DAY 1–31 chưa kiểm tra lịch theo tháng.

In [7]:
domains = {'YEAR':F.col('YEAR')==2015, 'MONTH':F.col('MONTH').between(1,12),
           'DAY':F.col('DAY').between(1,31), 'DAY_OF_WEEK':F.col('DAY_OF_WEEK').between(1,7),
           'DIVERTED':F.col('DIVERTED').isin(0,1), 'CANCELLED':F.col('CANCELLED').isin(0,1)}
domain_invalid = flights.agg(*[n_if(F.col(c).isNotNull() & ~valid).alias(c) for c,valid in domains.items()]).first().asDict()
table([{'column':c,'invalid_non_null':v,'null_count':flight_missing[c]['null_count']} for c,v in domain_invalid.items()])
coordinate_invalid = airports.agg(
    n_if(F.col('LATITUDE').isNotNull() & (~F.col('LATITUDE').between(-90,90) | F.isnan('LATITUDE'))).alias('LATITUDE'),
    n_if(F.col('LONGITUDE').isNotNull() & (~F.col('LONGITUDE').between(-180,180) | F.isnan('LONGITUDE'))).alias('LONGITUDE')
).first().asDict()
table([{**coord, **coordinate_invalid}])
reference_check('airport_exactly_one_coordinate_null', coord['one_null'], 0)

| column | invalid_non_null | null_count |
| --- | --- | --- |
| YEAR | 0 | 0 |
| MONTH | 0 | 0 |
| DAY | 0 | 0 |
| DAY_OF_WEEK | 0 | 0 |
| DIVERTED | 0 | 0 |
| CANCELLED | 0 | 0 |

| both_null | one_null | LATITUDE | LONGITUDE |
| --- | --- | --- | --- |
| 3 | 0 | 0 | 0 |

airport_exactly_one_coordinate_null: observed=0; reference=0; status=MATCH


## 8. HHMM Validity

Các cột giờ giữ nguyên string. Các nhóm loại trừ nhau: null, blank, HHMM hợp lệ 0000–2359 (phút ≤59), `2400`, sai định dạng, sai miền. `2400` là convention được báo riêng; không chuyển timestamp hay sửa chuỗi.

In [8]:
hhmm_cols = ['SCHEDULED_DEPARTURE','DEPARTURE_TIME','WHEELS_OFF','WHEELS_ON','SCHEDULED_ARRIVAL','ARRIVAL_TIME']
hhmm_exprs = []
for name in hhmm_cols:
    assert isinstance(flights.schema[name].dataType, T.StringType)
    c = F.col(name); fmt = c.rlike('^[0-9]{4}$')
    valid = fmt & c.substr(1,2).cast('int').between(0,23) & c.substr(3,2).cast('int').between(0,59)
    checks = {'null':c.isNull(), 'blank':c.isNotNull() & (F.trim(c)==''),
              'valid_hhmm':valid, 'value_2400':c=='2400',
              'invalid_format':c.isNotNull() & (F.trim(c)!='') & ~fmt,
              'invalid_range':fmt & ~valid & (c!='2400')}
    hhmm_exprs += [n_if(cond).alias(name+'__'+key) for key,cond in checks.items()]
hhmm_agg = flights.agg(*hhmm_exprs).first().asDict()
hhmm = [{'column':c, **{k:hhmm_agg[c+'__'+k] for k in checks}} for c in hhmm_cols]
table(hhmm)
assert all(sum(r[k] for k in checks) == N for r in hhmm)

| column | null | blank | valid_hhmm | value_2400 | invalid_format | invalid_range |
| --- | --- | --- | --- | --- | --- | --- |
| SCHEDULED_DEPARTURE | 0 | 0 | 5819079 | 0 | 0 | 0 |
| DEPARTURE_TIME | 86153 | 0 | 5732413 | 513 | 0 | 0 |
| WHEELS_OFF | 89047 | 0 | 5729305 | 727 | 0 | 0 |
| WHEELS_ON | 92513 | 0 | 5724561 | 2005 | 0 | 0 |
| SCHEDULED_ARRIVAL | 0 | 0 | 5819077 | 2 | 0 | 0 |
| ARRIVAL_TIME | 92513 | 0 | 5724110 | 2456 | 0 | 0 |

## 9. Cardinality

Dùng countDistinct **chính xác**, không tính null. Cardinality giúp module feature đánh giá số mức và rủi ro encoding; notebook chưa encoding.

In [9]:
card_cols = ['AIRLINE','FLIGHT_NUMBER','TAIL_NUMBER','ORIGIN_AIRPORT','DESTINATION_AIRPORT','CANCELLATION_REASON']
cardinality = flights.agg(*[F.countDistinct(c).alias(c) for c in card_cols]).first().asDict()
table([{'column':c, 'exact_distinct_non_null':v, 'null_count':flight_missing[c]['null_count']} for c,v in cardinality.items()])

| column | exact_distinct_non_null | null_count |
| --- | --- | --- |
| AIRLINE | 14 | 0 |
| FLIGHT_NUMBER | 6952 | 0 |
| TAIL_NUMBER | 4897 | 14721 |
| ORIGIN_AIRPORT | 628 | 0 |
| DESTINATION_AIRPORT | 629 | 0 |
| CANCELLATION_REASON | 4 | 5729195 |

## 10. Referential Integrity

Join với tập khóa lookup distinct để không nhân bản dòng. Coverage dùng tất cả flights làm mẫu số, kể cả khóa thiếu. Phân biệt IATA ba chữ in hoa với numeric năm chữ số (dạng DOT, chưa xác minh mapping). Thống kê theo tháng cho cả origin và destination; chỉ thu thập mã unmatched nếu có ≤20 mã.

In [10]:
ri = {}
for c, lookup in [('AIRLINE',airlines), ('ORIGIN_AIRPORT',airports), ('DESTINATION_AIRPORT',airports)]:
    keys = lookup.select(F.col('IATA_CODE').alias(c)).where(F.col(c).isNotNull()).distinct().withColumn('_matched', F.lit(1))
    joined = flights.select(c).join(F.broadcast(keys), c, 'left')
    matched = joined.agg(n_if(F.col('_matched')==1).alias('matched')).first()['matched']
    ri[c] = {'column':c,'total_rows':N,'matched_rows':matched,'unmatched_rows':N-matched,'coverage_pct':100*matched/N}
    unmatched_codes = joined.where(F.col('_matched').isNull()).select(c).distinct()
    code_count = unmatched_codes.count()
    print(c, 'distinct unmatched codes (including null if present):', code_count)
    if code_count <= 20: unmatched_codes.show(20, truncate=False)
    else: print('Code listing omitted: more than 20 distinct values.')
table(list(ri.values()))
month_exprs = []
for c in ['ORIGIN_AIRPORT','DESTINATION_AIRPORT']:
    month_exprs += [n_if(F.col(c).rlike('^[A-Z]{3}$')).alias(c+'_iata3'),
                   n_if(F.col(c).rlike('^[0-9]{5}$')).alias(c+'_numeric5'),
                   n_if(F.col(c).isNull()).alias(c+'_null'),
                   n_if(F.col(c).isNotNull() & ~F.col(c).rlike('^([A-Z]{3}|[0-9]{5})$')).alias(c+'_other')]
month_exprs += [n_if(F.col('ORIGIN_AIRPORT').rlike('^[0-9]{5}$') | F.col('DESTINATION_AIRPORT').rlike('^[0-9]{5}$')).alias('either_numeric_rows')]
monthly = [r.asDict() for r in flights.groupBy('MONTH').agg(F.count('*').alias('rows'), *month_exprs).orderBy('MONTH').collect()]
table(monthly)
month10 = next((r['either_numeric_rows'] for r in monthly if r['MONTH']==10), 0)
reference_check('month10_numeric_airport_rows', month10, 486165)
reference_check('airline_coverage_pct', ri['AIRLINE']['coverage_pct'], 100)
for c in ['ORIGIN_AIRPORT','DESTINATION_AIRPORT']:
    reference_check(c+'_coverage_pct', ri[c]['coverage_pct'], 91.645, 0.001)

AIRLINE distinct unmatched codes (including null if present): 0


+-------+
|AIRLINE|
+-------+
+-------+



ORIGIN_AIRPORT distinct unmatched codes (including null if present): 306
Code listing omitted: more than 20 distinct values.


DESTINATION_AIRPORT distinct unmatched codes (including null if present): 307
Code listing omitted: more than 20 distinct values.


| column | total_rows | matched_rows | unmatched_rows | coverage_pct |
| --- | --- | --- | --- | --- |
| AIRLINE | 5819079 | 5819079 | 0 | 100.0 |
| ORIGIN_AIRPORT | 5819079 | 5332914 | 486165 | 91.64532737912648 |
| DESTINATION_AIRPORT | 5819079 | 5332914 | 486165 | 91.64532737912648 |

| MONTH | rows | ORIGIN_AIRPORT_iata3 | ORIGIN_AIRPORT_numeric5 | ORIGIN_AIRPORT_null | ORIGIN_AIRPORT_other | DESTINATION_AIRPORT_iata3 | DESTINATION_AIRPORT_numeric5 | DESTINATION_AIRPORT_null | DESTINATION_AIRPORT_other | either_numeric_rows |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 469968 | 469968 | 0 | 0 | 0 | 469968 | 0 | 0 | 0 | 0 |
| 2 | 429191 | 429191 | 0 | 0 | 0 | 429191 | 0 | 0 | 0 | 0 |
| 3 | 504312 | 504312 | 0 | 0 | 0 | 504312 | 0 | 0 | 0 | 0 |
| 4 | 485151 | 485151 | 0 | 0 | 0 | 485151 | 0 | 0 | 0 | 0 |
| 5 | 496993 | 496993 | 0 | 0 | 0 | 496993 | 0 | 0 | 0 | 0 |
| 6 | 503897 | 503897 | 0 | 0 | 0 | 503897 | 0 | 0 | 0 | 0 |
| 7 | 520718 | 520718 | 0 | 0 | 0 | 520718 | 0 | 0 | 0 | 0 |
| 8 | 510536 | 510536 | 0 | 0 | 0 | 510536 | 0 | 0 | 0 | 0 |
| 9 | 464946 | 464946 | 0 | 0 | 0 | 464946 | 0 | 0 | 0 | 0 |
| 10 | 486165 | 0 | 486165 | 0 | 0 | 0 | 486165 | 0 | 0 | 486165 |
| 11 | 467972 | 467972 | 0 | 0 | 0 | 467972 | 0 | 0 | 0 | 0 |
| 12 | 479230 | 479230 | 0 | 0 | 0 | 479230 | 0 | 0 | 0 | 0 |

month10_numeric_airport_rows: observed=486165; reference=486165; status=MATCH
airline_coverage_pct: observed=100.0; reference=100; status=MATCH
ORIGIN_AIRPORT_coverage_pct: observed=91.64532737912648; reference=91.645; status=MATCH
DESTINATION_AIRPORT_coverage_pct: observed=91.64532737912648; reference=91.645; status=MATCH


## 11. Target & Eligibility

Regression target là ARRIVAL_DELAY. Eligibility yêu cầu target không null và cả CANCELLED, DIVERTED bằng 0; đây là phạm vi đánh giá outcome, chưa tạo dataset cho model.

In [11]:
eligible_condition = F.col('ARRIVAL_DELAY').isNotNull() & (F.col('CANCELLED')==0) & (F.col('DIVERTED')==0)
eligible_count = flights.agg(n_if(eligible_condition).alias('eligible')).first()['eligible']
target = {'total_rows':N, 'eligible':eligible_count, 'excluded':N-eligible_count,
          **{k:patterns[k] for k in ['arrival_delay_null','cancelled','diverted']}}
table([target])
reference_check('eligible_rows', eligible_count, 5714008)

| total_rows | eligible | excluded | arrival_delay_null | cancelled | diverted |
| --- | --- | --- | --- | --- | --- |
| 5819079 | 5714008 | 105071 | 105071 | 89884 | 15187 |

eligible_rows: observed=5714008; reference=5714008; status=MATCH


## 12. Classification Label Distribution

Trên eligible rows, dùng biểu thức tạm: positive nếu ARRIVAL_DELAY >30, negative nếu ≤30. Tỷ lệ lấy trên eligible rows để mô tả imbalance; không ghi nhãn vào dữ liệu.

In [12]:
label_stats = flights.where(eligible_condition).agg(
    n_if(F.col('ARRIVAL_DELAY') > 30).alias('positive'),
    n_if(F.col('ARRIVAL_DELAY') <= 30).alias('negative')).first().asDict()
label_stats['positive_rate_pct'] = 100*label_stats['positive']/eligible_count
table([label_stats])
reference_check('positive', label_stats['positive'], 632697)
reference_check('negative', label_stats['negative'], 5081311)
reference_check('positive_rate_pct', label_stats['positive_rate_pct'], 11.07, 0.01)

| positive | negative | positive_rate_pct |
| --- | --- | --- |
| 632697 | 5081311 | 11.072735634951858 |

positive: observed=632697; reference=632697; status=MATCH
negative: observed=5081311; reference=5081311; status=MATCH
positive_rate_pct: observed=11.072735634951858; reference=11.07; status=MATCH


## 13. Numeric Descriptive Statistics

Thống kê trên các giá trị non-null của toàn bảng flights, không chỉ eligible rows. Quartile dùng approxQuantile với relativeError=0.001 (sai số hạng). **ARRIVAL_DELAY âm hợp lệ về semantic: chuyến bay đến sớm**; không clip số âm. Tỷ lệ âm dùng số target non-null làm mẫu số.

In [13]:
numeric_cols = ['ARRIVAL_DELAY','SCHEDULED_TIME','DISTANCE','DEPARTURE_DELAY']
exprs = []
for c in numeric_cols:
    for key, func in [('count',F.count),('min',F.min),('max',F.max),('mean',F.mean),('stddev',F.stddev_samp)]:
        exprs.append(func(c).alias(c+'__'+key))
numeric_agg = flights.agg(*exprs, n_if(F.col('ARRIVAL_DELAY')<0).alias('arrival_negative')).first().asDict()
quantiles = flights.approxQuantile(numeric_cols, [0.25,0.5,0.75], 0.001)
numeric = []
for c, q in zip(numeric_cols,quantiles):
    numeric.append({'column':c, **{k:numeric_agg[c+'__'+k] for k in ['count','min','max','mean','stddev']},
                    **dict(zip(['q25_approx','q50_approx','q75_approx'],q))})
table(numeric)
negative_pct = 100*numeric_agg['arrival_negative']/numeric_agg['ARRIVAL_DELAY__count']
print('ARRIVAL_DELAY negative rate among non-null targets (%):', negative_pct)
for key, ref, tol in [('min',-87,0),('max',1971,0),('mean',4.407,0.001)]:
    reference_check('arrival_delay_'+key, numeric_agg['ARRIVAL_DELAY__'+key], ref, tol)
reference_check('arrival_delay_negative_pct', negative_pct, 61.27, 0.01)

| column | count | min | max | mean | stddev | q25_approx | q50_approx | q75_approx |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ARRIVAL_DELAY | 5714008 | -87 | 1971 | 4.407057357987598 | 39.27129709388725 | -13.0 | -5.0 | 8.0 |
| SCHEDULED_TIME | 5819073 | 18 | 718 | 141.6858915500802 | 75.21058230600217 | 85.0 | 123.0 | 174.0 |
| DISTANCE | 5819079 | 21 | 4983 | 822.3564947305235 | 607.7842873170591 | 373.0 | 647.0 | 1065.0 |
| DEPARTURE_DELAY | 5732926 | -82 | 1988 | 9.370158275198389 | 37.08094249678673 | -5.0 | -2.0 | 7.0 |

ARRIVAL_DELAY negative rate among non-null targets (%): 61.26871015931374
arrival_delay_min: observed=-87; reference=-87; status=MATCH
arrival_delay_max: observed=1971; reference=1971; status=MATCH
arrival_delay_mean: observed=4.407057357987598; reference=4.407; status=MATCH
arrival_delay_negative_pct: observed=61.26871015931374; reference=61.27; status=MATCH


## 14. Leakage & Inference Contract

Thời điểm inference: **before actual departure**. Bảng là hợp đồng documentation-level về khả năng sử dụng cột; candidate cần xác minh availability thực tế. Cờ eligibility chỉ dùng chọn quần thể có nhãn sau khi biết outcome, không dùng làm predictor.

In [14]:
candidates = ['YEAR','MONTH','DAY','DAY_OF_WEEK','AIRLINE','FLIGHT_NUMBER','ORIGIN_AIRPORT','DESTINATION_AIRPORT','SCHEDULED_DEPARTURE','SCHEDULED_ARRIVAL','SCHEDULED_TIME','DISTANCE']
leakage = ['DEPARTURE_TIME','DEPARTURE_DELAY','TAXI_OUT','WHEELS_OFF','ELAPSED_TIME','AIR_TIME','WHEELS_ON','TAXI_IN','ARRIVAL_TIME','CANCELLATION_REASON'] + causes
contract = ([{'Column':c,'Role':'CANDIDATE PRE-DEPARTURE','Reason':'Thông tin lịch/đường bay; xác minh availability lúc inference'} for c in candidates]
    + [{'Column':'TAIL_NUMBER','Role':'REQUIRES REVIEW','Reason':'Xác minh thời điểm biết và khả năng thay đổi tàu bay'},
       {'Column':'ARRIVAL_DELAY','Role':'TARGET','Reason':'Outcome sau chuyến bay'}]
    + [{'Column':c,'Role':'ELIGIBILITY ONLY','Reason':'Cờ outcome; không là predictor'} for c in ['CANCELLED','DIVERTED']]
    + [{'Column':c,'Role':'LEAKAGE / POST-OUTCOME','Reason':'Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome'} for c in leakage])
assert set(r['Column'] for r in contract) == set(flights.columns)
table(contract)

| Column | Role | Reason |
| --- | --- | --- |
| YEAR | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| MONTH | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| DAY | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| DAY_OF_WEEK | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| AIRLINE | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| FLIGHT_NUMBER | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| ORIGIN_AIRPORT | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| DESTINATION_AIRPORT | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| SCHEDULED_DEPARTURE | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| SCHEDULED_ARRIVAL | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| SCHEDULED_TIME | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| DISTANCE | CANDIDATE PRE-DEPARTURE | Thông tin lịch/đường bay; xác minh availability lúc inference |
| TAIL_NUMBER | REQUIRES REVIEW | Xác minh thời điểm biết và khả năng thay đổi tàu bay |
| ARRIVAL_DELAY | TARGET | Outcome sau chuyến bay |
| CANCELLED | ELIGIBILITY ONLY | Cờ outcome; không là predictor |
| DIVERTED | ELIGIBILITY ONLY | Cờ outcome; không là predictor |
| DEPARTURE_TIME | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| DEPARTURE_DELAY | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| TAXI_OUT | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| WHEELS_OFF | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| ELAPSED_TIME | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| AIR_TIME | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| WHEELS_ON | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| TAXI_IN | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| ARRIVAL_TIME | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| CANCELLATION_REASON | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| AIR_SYSTEM_DELAY | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| SECURITY_DELAY | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| AIRLINE_DELAY | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| LATE_AIRCRAFT_DELAY | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |
| WEATHER_DELAY | LEAKAGE / POST-OUTCOME | Thông tin thực tế trong/sau hành trình hoặc nguyên nhân outcome |

## 15. Missing-Data Decision Matrix

Ghi nhận bằng chứng và đề xuất cho module tiếp theo, chưa implement treatment. Liên hệ missing/outcome không đủ để tự động fill 0 hoặc suy ra cơ chế missing.

In [15]:
def decision(group, finding, interpretation, impact, action, owner):
    return dict(zip(['Column/group','Finding','Interpretation','Impact','Recommended next action','Owner'],
                    [group,finding,interpretation,impact,action,owner]))
actual_cols = ['DEPARTURE_TIME','WHEELS_OFF','WHEELS_ON','ARRIVAL_TIME','ELAPSED_TIME','AIR_TIME']
decisions = [
 decision('ARRIVAL_DELAY', f"{patterns['arrival_delay_null']} null; invariant exceptions={patterns['null_without_flag']+patterns['flag_without_null']}", 'Thiếu target liên hệ với cờ outcome theo bảng chéo', 'Không có nhãn cho một phần flights', 'Áp dụng eligibility đã thống nhất trong pipeline sau; giữ raw', 'Pipeline'),
 decision('LATITUDE/LONGITUDE', f"both null={coord['both_null']}; one null={coord['one_null']}", 'Lookup thiếu tọa độ, chưa xác minh nguyên nhân', 'Feature địa lý có thể thiếu', 'Xác minh nguồn tọa độ và chính sách missing', 'Custom/Feature'),
 decision('Delay-cause columns', f"Missing {min(r['missing_rate_pct'] for r in cause_rows):.3f}–{max(r['missing_rate_pct'] for r in cause_rows):.3f}%", 'Kiểm tra liên hệ với target ≥15 tại section 5', 'Post-outcome leakage', 'Loại khỏi candidate inference; không tự fill 0', 'Custom/Feature'),
 decision('TAIL_NUMBER', f"Null={flight_missing['TAIL_NUMBER']['null_count']}; distinct={cardinality['TAIL_NUMBER']}", 'Availability trước departure cần review', 'Missing và high cardinality', 'Quyết định sử dụng sau khi xác minh availability', 'Custom/Feature'),
 decision('Actual-time columns', '; '.join(f"{c}: {flight_missing[c]['null_rate_pct']:.3f}% null" for c in actual_cols), 'Giá trị thực tế có thiếu; nguyên nhân chưa kết luận', 'Không có sẵn tại inference trước departure', 'Giữ để audit; loại khỏi predictors theo contract', 'Pipeline')]
table(decisions)

| Column/group | Finding | Interpretation | Impact | Recommended next action | Owner |
| --- | --- | --- | --- | --- | --- |
| ARRIVAL_DELAY | 105071 null; invariant exceptions=0 | Thiếu target liên hệ với cờ outcome theo bảng chéo | Không có nhãn cho một phần flights | Áp dụng eligibility đã thống nhất trong pipeline sau; giữ raw | Pipeline |
| LATITUDE/LONGITUDE | both null=3; one null=0 | Lookup thiếu tọa độ, chưa xác minh nguyên nhân | Feature địa lý có thể thiếu | Xác minh nguồn tọa độ và chính sách missing | Custom/Feature |
| Delay-cause columns | Missing 81.725–81.725% | Kiểm tra liên hệ với target ≥15 tại section 5 | Post-outcome leakage | Loại khỏi candidate inference; không tự fill 0 | Custom/Feature |
| TAIL_NUMBER | Null=14721; distinct=4897 | Availability trước departure cần review | Missing và high cardinality | Quyết định sử dụng sau khi xác minh availability | Custom/Feature |
| Actual-time columns | DEPARTURE_TIME: 1.481% null; WHEELS_OFF: 1.530% null; WHEELS_ON: 1.590% null; ARRIVAL_TIME: 1.590% null; ELAPSED_TIME: 1.806% null; AIR_TIME: 1.806% null | Giá trị thực tế có thiếu; nguyên nhân chưa kết luận | Không có sẵn tại inference trước departure | Giữ để audit; loại khỏi predictors theo contract | Pipeline |

## 16. Data Quality Scorecard

Scorecard dựa trên kết quả notebook, không ép PASS. Accuracy cấp record cần ground truth ngoài dataset; timeliness bị giới hạn bởi dữ liệu năm 2015. Uniqueness của flights chưa được kiểm tra trong notebook.

In [16]:
invalid_hhmm = sum(r['invalid_format']+r['invalid_range'] for r in hhmm)
validity_failures = sum(domain_invalid.values())+sum(coordinate_invalid.values())+invalid_hhmm
consistency_exceptions = patterns['null_without_flag']+patterns['flag_without_null']+patterns['unknown_flags']+coord['one_null']
lookup_duplicates = sum(r['duplicate_key_count'] for r in uniqueness)
score_rows = [
 {'Dimension':'Completeness','Status':'PARTIAL' if any(r['null_count'] or (isinstance(r['blank_count'],int) and r['blank_count']) for rows in completeness.values() for r in rows) else 'PASS','Evidence':f"Target null={patterns['arrival_delay_null']}; coordinates both null={coord['both_null']}",'Limitation / next action':'Áp dụng decision matrix ở module sau'},
 {'Dimension':'Validity','Status':'PASS' if validity_failures==0 else 'FAIL','Evidence':f'Domain/HHMM/coordinate violations={validity_failures}','Limitation / next action':'Chỉ các domain đã kiểm tra; chưa calendar/chronology validation'},
 {'Dimension':'Uniqueness','Status':'PARTIAL' if lookup_duplicates==0 else 'FAIL','Evidence':f'Lookup duplicate keys={lookup_duplicates}; flights full-row SKIPPED','Limitation / next action':'Không kết luận uniqueness cho flights'},
 {'Dimension':'Consistency','Status':'PASS' if consistency_exceptions==0 else 'FAIL','Evidence':f'Checked missing-pattern exceptions={consistency_exceptions}','Limitation / next action':'Chỉ target/flags và cặp tọa độ; chưa kiểm tra mọi quan hệ'},
 {'Dimension':'Referential Integrity','Status':'PARTIAL' if any(r['unmatched_rows'] for r in ri.values()) else 'PASS','Evidence':'; '.join(f"{c}={r['coverage_pct']:.6f}%" for c,r in ri.items()),'Limitation / next action':'Xác minh DOT → IATA mapping'},
 {'Dimension':'Accuracy','Status':'UNASSESSED at record level','Evidence':'Không có external ground truth','Limitation / next action':'Cần nguồn đối soát độc lập'},
 {'Dimension':'Timeliness','Status':'LIMITATION','Evidence':'Dataset năm 2015','Limitation / next action':'Chưa đại diện vận hành hiện tại'}]
table(score_rows)
scorecard = {r['Dimension']:r['Status'] for r in score_rows}

| Dimension | Status | Evidence | Limitation / next action |
| --- | --- | --- | --- |
| Completeness | PARTIAL | Target null=105071; coordinates both null=3 | Áp dụng decision matrix ở module sau |
| Validity | PASS | Domain/HHMM/coordinate violations=0 | Chỉ các domain đã kiểm tra; chưa calendar/chronology validation |
| Uniqueness | PARTIAL | Lookup duplicate keys=0; flights full-row SKIPPED | Không kết luận uniqueness cho flights |
| Consistency | PASS | Checked missing-pattern exceptions=0 | Chỉ target/flags và cặp tọa độ; chưa kiểm tra mọi quan hệ |
| Referential Integrity | PARTIAL | AIRLINE=100.000000%; ORIGIN_AIRPORT=91.645327%; DESTINATION_AIRPORT=91.645327% | Xác minh DOT → IATA mapping |
| Accuracy | UNASSESSED at record level | Không có external ground truth | Cần nguồn đối soát độc lập |
| Timeliness | LIMITATION | Dataset năm 2015 | Chưa đại diện vận hành hiện tại |

## 17. Conclusion & Handoff

Typed Parquet đọc được và có cấu trúc hợp lệ trong phạm vi các kiểm tra phía trên. **Không có cast-loss đã biết theo kết quả prepare_data.py được bàn giao (`RESULT|PASS`)**; notebook không đọc lại CSV để tái xác minh cast-loss. Các vấn đề chính là target/missing theo outcome, tọa độ thiếu, delay-cause leakage và airport numeric dạng DOT không match trực tiếp IATA. DOT mapping và missing treatment chưa thực hiện; dữ liệu **chưa cleaned/model-ready**.

| Module nhận | Handoff |
|---|---|
| Custom/Feature | DOT airport mapping; missing coordinates; feature decisions; leakage list và TAIL_NUMBER review |
| Pipeline | Typed schema; target eligibility; inference contract trước actual departure |
| Model | Regression target âm hợp lệ; classification imbalance; dùng identical split để so sánh mô hình công bằng ở module sau |
| Production | Input schema; hai dạng IATA/DOT; kiểm tra tương thích schema JSON streaming trong tương lai |

In [17]:
summary = {
 'DATASETS': counts,
 'COMPLETENESS': {'arrival_delay_null':patterns['arrival_delay_null'], 'airport_both_coordinates_null':coord['both_null'],
                  'delay_cause_missing_rate_range':[min(r['missing_rate_pct'] for r in cause_rows),max(r['missing_rate_pct'] for r in cause_rows)]},
 'VALIDITY': {'invalid_domain_checks':domain_invalid,'invalid_hhmm_counts':{r['column']:r['invalid_format']+r['invalid_range'] for r in hhmm},'coordinate_range_violations':coordinate_invalid},
 'UNIQUENESS': {r['table']+'_iata_duplicate_keys':r['duplicate_key_count'] for r in uniqueness},
 'REFERENTIAL_INTEGRITY': {**{c:r['coverage_pct'] for c,r in ri.items()},'month10_numeric_airport_rows':month10},
 'TARGET': {'eligible_rows':eligible_count,**label_stats},
 'NUMERIC_PROFILE': {**{k:numeric_agg['ARRIVAL_DELAY__'+k] for k in ['min','max','mean']},'arrival_delay_negative_pct':negative_pct},
 'SCORECARD':scorecard,
 'SKIPPED_CHECKS':['Exact full-row duplicate: previously profiled on raw data (handoff context); not rerun here'],
 'WARNINGS_OR_DISCREPANCIES':warnings}
summary['UNIQUENESS']['full_row_duplicate_status'] = full_duplicate_status
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "DATASETS": {
    "flights": 5819079,
    "airports": 322,
    "airlines": 14
  },
  "COMPLETENESS": {
    "arrival_delay_null": 105071,
    "airport_both_coordinates_null": 3,
    "delay_cause_missing_rate_range": [
      81.72496025573807,
      81.72496025573807
    ]
  },
  "VALIDITY": {
    "invalid_domain_checks": {
      "YEAR": 0,
      "MONTH": 0,
      "DAY": 0,
      "DAY_OF_WEEK": 0,
      "DIVERTED": 0,
      "CANCELLED": 0
    },
    "invalid_hhmm_counts": {
      "SCHEDULED_DEPARTURE": 0,
      "DEPARTURE_TIME": 0,
      "WHEELS_OFF": 0,
      "WHEELS_ON": 0,
      "SCHEDULED_ARRIVAL": 0,
      "ARRIVAL_TIME": 0
    },
    "coordinate_range_violations": {
      "LATITUDE": 0,
      "LONGITUDE": 0
    }
  },
  "UNIQUENESS": {
    "airports_iata_duplicate_keys": 0,
    "airlines_iata_duplicate_keys": 0,
    "full_row_duplicate_status": "SKIPPED"
  },
  "REFERENTIAL_INTEGRITY": {
    "AIRLINE": 100.0,
    "ORIGIN_AIRPORT": 91.64532737912648,
    "DESTINATION_AIRPORT": 9

## 18. Cleanup

Đóng SparkSession và giải phóng tài nguyên sau profiling. Notebook không persist DataFrame và không ghi dataset.

In [18]:
spark.stop()
print('Spark stopped. Profiling complete; no dataset written.')

Spark stopped. Profiling complete; no dataset written.
